In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)


In [2]:
df = pd.read_csv(r'C:\Users\orams\Documents\Customer-Churn-Prediction\data\processed_data\telco_churn_ml_ready.csv')

In [3]:
X = df.drop(columns=['Churn'])
y =  df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Using random_state= 42 and stratify=y so it corresponds to Week 3 works.

In [4]:
#S2. Task 1 Class Imbalance Analysis

In [5]:
counts = y_train.value_counts()
print(counts)
print("Imbalance ratio:", round(counts['No'] / counts['Yes'], 2))

Churn
No     4139
Yes    1495
Name: count, dtype: int64
Imbalance ratio: 2.77


The majority class is Non-churners(No), counts 4139 customers as compared to 1495 churners(Yes) giving an imbalance ratio 2.77 non churners for every churner. This case can cause a model to favour the majority class and perform poorly at identifying churners. Week 3, the models had recall scores between 47% and 52%, meaning they missed roughly 48% to 53% of actual churners. This is a concern because missed churners represent customers the business may lose without having a chance to intervene.

In [6]:
#S3. Task 2 First Imbalance Technique

In [7]:
log_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
log_bal.fit(X_train, y_train)

y_pred_bal = log_bal.predict(X_test)
y_proba_bal = log_bal.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_bal))
print("Precision:", precision_score(y_test, y_pred_bal, pos_label='Yes'))
print("Recall:", recall_score(y_test, y_pred_bal, pos_label='Yes'))
print("F1-score:", f1_score(y_test, y_pred_bal, pos_label='Yes'))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_bal))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bal))

Accuracy: 0.7324343506032647
Precision: 0.49747899159663866
Recall: 0.7914438502673797
F1-score: 0.6109391124871001
ROC-AUC: 0.8421271538918598

Confusion Matrix:
 [[736 299]
 [ 78 296]]


Choose the classs weighted Logistic Regression model becasue false negatives are more costly than false positives in this situation. The model increased recall from 51.9% to 79.1%, reducing missed churners from 180 to 78.  This created 197 additional false positives, where those customers would receive a retention offer while a false negative represents a customer who may leave without intervention. The improvement in F1-score from 57.9% to 61.1% supports the trade-off. I would prefer the weighted model despite its lower accuracy and precision.

In [8]:
#Random Forest With Class Weighting

In [9]:
rf_bal = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_bal.fit(X_train, y_train)

y_pred_rf_bal = rf_bal.predict(X_test)
y_proba_rf_bal = rf_bal.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_rf_bal))
print("Precision:", precision_score(y_test, y_pred_rf_bal, pos_label='Yes'))
print("Recall:", recall_score(y_test, y_pred_rf_bal, pos_label='Yes'))
print("F1-score:", f1_score(y_test, y_pred_rf_bal, pos_label='Yes'))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf_bal))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf_bal))

Accuracy: 0.7643718949609652
Precision: 0.547945205479452
Recall: 0.6417112299465241
F1-score: 0.5911330049261084
ROC-AUC: 0.8221434291766772

Confusion Matrix:
 [[837 198]
 [134 240]]


In [10]:
#Second Technique OVERSAMPLING

In [11]:
!pip install imbalanced-learn

In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(y_train_sm.value_counts())

Churn
No     4139
Yes    4139
Name: count, dtype: int64


Only resample X_train, y_train not the test set becasue the test should represent the original real-world data distribution.

In [13]:
log_sm = LogisticRegression(max_iter=1000, random_state=42)
log_sm.fit(X_train_sm, y_train_sm)

y_pred_sm = log_sm.predict(X_test)
y_proba_sm = log_sm.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_sm))
print("Precision:", precision_score(y_test, y_pred_sm, pos_label='Yes'))
print("Recall:", recall_score(y_test, y_pred_sm, pos_label='Yes'))
print("F1-score:", f1_score(y_test, y_pred_sm, pos_label='Yes'))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_sm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_sm))

Accuracy: 0.758694109297374
Precision: 0.5354166666666667
Recall: 0.6871657754010695
F1-score: 0.6018735362997658
ROC-AUC: 0.8286755018212818

Confusion Matrix:
 [[812 223]
 [117 257]]


Keep Class weighting for Logistic Regression. This gives the highestr recall at 79.1% meaning it misses fewer actual churners than SMOTE at 68.7%, best F1 score at 61.1% , slightly higher than SMOTE's 60.2%. Its ROC-AUC of 84.21% is almost identical to the baseline 84.24%, while SMOTE drops to 82.87%.

In [14]:
#Task3 Cross-Validation. 

In [15]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# y must be numeric for these scorers, so map it
y_num = y.map({'No': 0, 'Yes': 1})

log_bal_cv = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
results = cross_validate(log_bal_cv, X, y_num, cv=cv, scoring=scoring)

for metric in scoring:
    scores = results[f'test_{metric}']
    print(metric, np.round(scores, 3), "mean:", round(scores.mean(), 3), "std:", round(scores.std(), 3))

accuracy [0.754 0.74  0.754 0.733 0.746] mean: 0.745 std: 0.008
precision [0.523 0.506 0.524 0.497 0.514] mean: 0.513 std: 0.01
recall [0.826 0.813 0.807 0.767 0.797] mean: 0.802 std: 0.02
f1 [0.64  0.624 0.636 0.603 0.625] mean: 0.626 std: 0.013
roc_auc [0.855 0.846 0.861 0.827 0.839] mean: 0.846 std: 0.012


The class-weighted Logistic Regression generalizes consistently, with mean recall 80.2% (std 0.02) and mean ROC-AUC 84.6% (std 0.012).

In [16]:
#Task4 Hyperparameter tuning


Decision Tree (GridSearchCV)

In [20]:
from sklearn.model_selection import GridSearchCV

y_train_num = y_train.map({'No': 0, 'Yes': 1})
y_test_num = y_test.map({'No': 0, 'Yes': 1})

dt_params = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_leaf': [10, 20, 50],
    'min_samples_split': [2, 20, 50]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    dt_params, cv=5, scoring='f1', n_jobs=-1
)
dt_grid.fit(X_train, y_train_num)

print("Best params:", dt_grid.best_params_)
print("Best CV F1:", round(dt_grid.best_score_, 3))

best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test_num, y_pred_dt))
print("Precision:", precision_score(y_test_num, y_pred_dt))
print("Recall:", recall_score(y_test_num, y_pred_dt))
print("F1-score:", f1_score(y_test_num, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test_num, y_proba_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_num, y_pred_dt))

Best params: {'max_depth': 7, 'min_samples_leaf': 50, 'min_samples_split': 2}
Best CV F1: 0.629
Accuracy: 0.7423704755145494
Precision: 0.5092436974789916
Recall: 0.8101604278074866
F1-score: 0.6253869969040248
ROC-AUC: 0.837518406572115

Confusion Matrix:
 [[743 292]
 [ 71 303]]


The five cross-validation folds look reasonably consistent, with relatively small standard deviations across the metrics. Recall ranges from 76.7% to 82.6%, with a mean of 80.2%, while F1 ranges from 60.3% to 64.0%, with a mean of 62.6%. These results are close to the single test split, where recall was 79.1% and F1 was 61.1%. This suggests that the class-weighted Logistic Regression performs consistently across different subsets of the data.

In [ ]:
#Random Forest(RandomizedSearchCV)

In [19]:
from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 12, None],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2']
}

predict Random Forest will gain less from tuning than the Decision Tree.

In [21]:
rf_grid = RandomizedSearchCV(
    RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    ),
    rf_params,
    n_iter=15,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train_num)

print("Best params:", rf_grid.best_params_)
print("Best CV F1:", round(rf_grid.best_score_, 3))

best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test_num, y_pred_rf))
print("Precision:", precision_score(y_test_num, y_pred_rf))
print("Recall:", recall_score(y_test_num, y_pred_rf))
print("F1-score:", f1_score(y_test_num, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test_num, y_proba_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_num, y_pred_rf))

Best params: {'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': None}
Best CV F1: 0.634
Accuracy: 0.7530163236337828
Precision: 0.5232974910394266
Recall: 0.7807486631016043
F1-score: 0.6266094420600858
ROC-AUC: 0.8418210235345785

Confusion Matrix:
 [[769 266]
 [ 82 292]]


In [22]:
#Cross Validation on the tuned random Forest
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
y_num = y.map({'No': 0, 'Yes': 1})

rf_results = cross_validate(best_rf, X, y_num, cv=cv, scoring=scoring)

for metric in scoring:
    scores = rf_results[f'test_{metric}']
    print(metric, np.round(scores, 3), "mean:", round(scores.mean(), 3), "std:", round(scores.std(), 3))

accuracy [0.77  0.749 0.779 0.75  0.755] mean: 0.76 std: 0.012
precision [0.546 0.518 0.559 0.52  0.528] mean: 0.534 std: 0.016
recall [0.786 0.759 0.781 0.745 0.735] mean: 0.761 std: 0.02
f1 [0.645 0.616 0.652 0.612 0.615] mean: 0.628 std: 0.017
roc_auc [0.859 0.838 0.857 0.825 0.839] mean: 0.844 std: 0.013


In [23]:
#Final Comparison Table
def evaluate(name, model, X_te, y_te_num):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_te_num, pred), 4),
        'Precision': round(precision_score(y_te_num, pred), 4),
        'Recall': round(recall_score(y_te_num, pred), 4),
        'F1-Score': round(f1_score(y_te_num, pred), 4),
        'ROC-AUC': round(roc_auc_score(y_te_num, proba), 4)
    }

y_train_num = y_train.map({'No': 0, 'Yes': 1})

lr_base = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train_num)
lr_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced').fit(X_train, y_train_num)

final_table = pd.DataFrame([
    evaluate('Logistic Regression (baseline)', lr_base, X_test, y_test_num),
    evaluate('Logistic Regression (class weighted)', lr_bal, X_test, y_test_num),
    evaluate('Decision Tree (tuned)', best_dt, X_test, y_test_num),
    evaluate('Random Forest (tuned)', best_rf, X_test, y_test_num),
])
final_table

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression (baseline),0.7999,0.6554,0.5187,0.5791,0.8424
1,Logistic Regression (class weighted),0.7324,0.4975,0.7914,0.6109,0.8421
2,Decision Tree (tuned),0.7424,0.5092,0.8102,0.6254,0.8375
3,Random Forest (tuned),0.7530,0.5233,0.7807,0.6266,0.8418


In [ ]:
#Final Model Selection

*LOGISTIC REGRESSION*

Logistic Regression achieved a mean cross-validated recall of 80.2% and a mean ROC-AUC of 84.6%, compared with Random Forest's 76.1% and 84.4%. Random Forest had a slightly higher F1-score (62.8% versus 62.6%), but this difference is insignificant, and both models showed similar reliability across the validation folds.

Recall is the deciding metric. A missed churner is the most costly error for the business, and the recall gap of about 4 points between the two models is roughly two standard deviations in cross-validation, which is large enough to matter. Logistic Regression therefore catches more of the customers who are actually about to leave.

The tuned Decision Tree had the highest test recall (81.0%), but it was not selected. It was evaluated on a single train-test split only, its ROC-AUC (83.8%) was the lowest of the three finalists, and a single tree is generally less stable than the other two models.

In terms of commercial applicability, Logistic Regression is cheaper to run and easier to explain than Random Forest. Its coefficients show which customer characteristics increase or decrease churn risk, which supports the feature importance analysis and business recommendations in Week 5.